In [10]:
import numpy as np
import pandas as pd
from tensorflow import keras
from sklearn.metrics import accuracy_score

In [4]:
class NN:
    def __init__(self, input_size=784, hidden_nodes=128, output_size=10):
        self.weights1 = np.random.randn(input_size, hidden_nodes) * np.sqrt(2 / input_size)
        self.weights2 = np.random.randn(hidden_nodes, output_size) * np.sqrt(2 / hidden_nodes)
        self.bias1 = np.zeros((1, hidden_nodes))
        self.bias2 = np.zeros((1, output_size))

    def relu(self, z):
        return np.maximum(0, z)

    def softmax(self, z):
        z = z - np.max(z, axis=1, keepdims=True)
        exp_z = np.exp(z)
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)

    def forward(self, x):
        self.x = x
        self.z1 = x @ self.weights1 + self.bias1
        self.a1 = self.relu(self.z1)
        self.z2 = self.a1 @ self.weights2 + self.bias2
        self.a2 = self.softmax(self.z2)
        return self.a2

    def cross_entropy(self, pred, y):
        epsilon = 1e-10
        correct_probs = pred[np.arange(len(y)), y]
        loss = -np.mean(np.log(correct_probs + epsilon))
        return loss

    def accuracy(self, pred, y):
        predictions = np.argmax(pred, axis=1)
        return np.mean(predictions == y)
    
    def back(self, y, lr=0.01):
        m = self.x.shape[0]
        dZ2 = self.a2.copy()
        dZ2[np.arange(m), y] -= 1
        dZ2 /= m
        dW2 = self.a1.T @ dZ2
        db2 = np.sum(dZ2, axis=0, keepdims=True)
        dA1 = dZ2 @ self.weights2.T
        dZ1 = dA1 * (self.z1 > 0)
        dW1 = self.x.T @ dZ1
        db1 = np.sum(dZ1, axis=0, keepdims=True)
        self.weights1 -= lr * dW1
        self.bias1 -= lr * db1
        self.weights2 -= lr * dW2
        self.bias2 -= lr * db2


    def train(self, x, y, epochs=100, lr=0.01):
        for epoch in range(epochs):
            pred = self.forward(x)
            loss = self.cross_entropy(pred, y)
            acc = self.accuracy(pred, y)
            self.back(y, lr)
            print(f"Epoch {epoch+1}/{epochs} | "f"Loss: {loss:.4f} | "f"Accuracy: {acc:.4f}")

    def predict(self, x):
        pred = self.forward(x)
        return np.argmax(pred, axis=1)

In [5]:
(x_train,y_train),(x_test,y_test)=keras.datasets.mnist.load_data()

In [6]:
x_train=x_train/255
x_test=x_test/255
x_train = x_train.reshape(x_train.shape[0], -1)
x_test = x_test.reshape(x_test.shape[0], -1)

In [7]:
print(x_train.shape,y_train.shape,x_test.shape,y_test.shape)

(60000, 784) (60000,) (10000, 784) (10000,)


In [8]:
model=NN()
model.train(x_train,y_train,epochs=200,lr=0.1)

Epoch 1/200 | Loss: 2.6150 | Accuracy: 0.0897
Epoch 2/200 | Loss: 2.3978 | Accuracy: 0.1018
Epoch 3/200 | Loss: 2.2780 | Accuracy: 0.1371
Epoch 4/200 | Loss: 2.1800 | Accuracy: 0.1984
Epoch 5/200 | Loss: 2.0929 | Accuracy: 0.2674
Epoch 6/200 | Loss: 2.0124 | Accuracy: 0.3389
Epoch 7/200 | Loss: 1.9363 | Accuracy: 0.4144
Epoch 8/200 | Loss: 1.8636 | Accuracy: 0.4900
Epoch 9/200 | Loss: 1.7938 | Accuracy: 0.5591
Epoch 10/200 | Loss: 1.7267 | Accuracy: 0.6091
Epoch 11/200 | Loss: 1.6622 | Accuracy: 0.6448
Epoch 12/200 | Loss: 1.6005 | Accuracy: 0.6723
Epoch 13/200 | Loss: 1.5414 | Accuracy: 0.6932
Epoch 14/200 | Loss: 1.4851 | Accuracy: 0.7090
Epoch 15/200 | Loss: 1.4316 | Accuracy: 0.7226
Epoch 16/200 | Loss: 1.3808 | Accuracy: 0.7340
Epoch 17/200 | Loss: 1.3328 | Accuracy: 0.7433
Epoch 18/200 | Loss: 1.2875 | Accuracy: 0.7516
Epoch 19/200 | Loss: 1.2448 | Accuracy: 0.7590
Epoch 20/200 | Loss: 1.2046 | Accuracy: 0.7655
Epoch 21/200 | Loss: 1.1669 | Accuracy: 0.7715
Epoch 22/200 | Loss: 1

In [9]:
y_pred=model.predict(x_test)

In [11]:
accuracy_score(y_test,y_pred)

0.905